# Hatch — Notebook 03: Model Training and INT8 Quantization

**Goal:** Train a small 1D-CNN that meets all three of:

1. ≥ 90% test-set accuracy on 4-class HumBug classification (target).
2. ≤ 50 KB INT8 model footprint (so the model + working buffers fit in ESP32-S3 SRAM).
3. ≤ 200 ms inference latency on ESP32-S3 @ 240 MHz.

**Architecture:** 4 separable-conv blocks (channels 16/32/64/128) → global average pool → 4-way softmax. Inspired by MobileNet's depthwise-separable convolutions and the MosquitoSong+ paper's noise-robust CNN.

**Quantization:** Post-training INT8 with TensorFlow Lite, calibrated on a held-out subset of the training data.

---

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='ticks', context='notebook')
plt.rcParams['figure.dpi'] = 110

FEAT_DIR = Path('../data/features/')
MODEL_DIR = Path('../models/')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')

## 1. Load precomputed features

In [ ]:
def load_split(name):
    z = np.load(FEAT_DIR / f'{name}.npz')
    return z['features'].astype(np.float32), z['labels'].astype(np.int32)

X_train, y_train = load_split('train')
X_val,   y_val   = load_split('val')
X_test,  y_test  = load_split('test')

print(f'Train: {X_train.shape}, labels: {np.bincount(y_train)}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

# Add channel dimension for Conv2D (treat mel-spec as single-channel image)
X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

N_MELS, N_FRAMES = X_train.shape[1], X_train.shape[2]
N_CLASSES = 4

## 2. Class weights

Because the corpus is imbalanced and the *Aedes aegypti* class is most operationally important, we use sklearn-style inverse-frequency class weights.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', class_weight_dict)

## 3. Architecture

Small separable-conv CNN. Stride-2 downsampling cuts feature-map size aggressively; global average pooling avoids the parameter-heavy dense layers that would blow the size budget.

In [ ]:
def build_model(n_mels=N_MELS, n_frames=N_FRAMES, n_classes=N_CLASSES):
    inputs = layers.Input(shape=(n_mels, n_frames, 1))
    
    x = layers.Conv2D(16, kernel_size=3, padding='same', strides=2)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(max_value=6.0)(x)   # ReLU6 quantizes better than plain ReLU
    
    for ch in [32, 64, 128]:
        x = layers.SeparableConv2D(ch, kernel_size=3, padding='same', strides=2)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6.0)(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    
    model = models.Model(inputs, out, name='hatch_acoustic_v01')
    return model

model = build_model()
model.summary()

**Parameter check.** The model should report on the order of 50–70k parameters. At INT8 (1 byte/param) plus activation buffers, this comfortably fits the 40 KB target.

## 4. Training

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cbs = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    callbacks.ModelCheckpoint(str(MODEL_DIR / 'best_fp32.keras'), save_best_only=True, monitor='val_accuracy')
]

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=cbs,
    verbose=2
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist.history['loss'], label='train')
axes[0].plot(hist.history['val_loss'], label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()
axes[1].plot(hist.history['accuracy'], label='train')
axes[1].plot(hist.history['val_accuracy'], label='val')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy'); axes[1].legend()
plt.tight_layout()

## 5. Test-set evaluation

In [ ]:
LABELS = ['noise', 'other_insect', 'ae_aegypti', 'ae_albopictus']

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=LABELS, digits=3))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Confusion matrix — HumBug test split (FP32)')
plt.tight_layout()

## 6. Post-training INT8 quantization

TensorFlow Lite's full-integer quantization converts weights and activations to INT8 using a small calibration dataset to learn per-tensor scale and zero-point.

In [ ]:
def representative_dataset():
    idx = np.random.choice(len(X_train), 200, replace=False)
    for i in idx:
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()

model_path = MODEL_DIR / 'hatch_acoustic_int8.tflite'
model_path.write_bytes(tflite_int8)
print(f'INT8 model size: {len(tflite_int8) / 1024:.1f} KB')
assert len(tflite_int8) < 60 * 1024, 'Model exceeds 60 KB budget — re-tune architecture'

## 7. Quantized accuracy check

Verify the INT8 model retains the FP32 model's accuracy (target: < 2 percentage-point drop).

In [ ]:
interp = tf.lite.Interpreter(model_path=str(model_path))
interp.allocate_tensors()
in_det  = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
in_scale, in_zp   = in_det['quantization']
out_scale, out_zp = out_det['quantization']

def int8_predict_one(x_float):
    x_q = np.round(x_float / in_scale + in_zp).clip(-128, 127).astype(np.int8)
    interp.set_tensor(in_det['index'], x_q)
    interp.invoke()
    y_q = interp.get_tensor(out_det['index'])
    return (y_q.astype(np.float32) - out_zp) * out_scale

y_pred_int8 = np.zeros(len(X_test), dtype=np.int32)
for i in range(len(X_test)):
    probs = int8_predict_one(X_test[i:i+1])
    y_pred_int8[i] = probs.argmax()

from sklearn.metrics import accuracy_score
acc_fp32 = (y_pred == y_test).mean()
acc_int8 = (y_pred_int8 == y_test).mean()
print(f'FP32 accuracy: {acc_fp32:.4f}')
print(f'INT8 accuracy: {acc_int8:.4f}')
print(f'Quantization loss: {(acc_fp32 - acc_int8) * 100:.2f} pp')

## 8. ESP32-S3 latency benchmark

Run on actual hardware using the Edge Impulse exported library. The benchmark below is the protocol; the actual numbers require an ESP32-S3 board attached.

```bash
# From /firmware:
pio run -t upload -e xiao_esp32s3
pio device monitor -b 115200
# Trigger benchmark mode via serial command 'BENCH'
# Expected output:
#   [bench] inference 1: 183 ms
#   [bench] inference 2: 181 ms
#   ...
#   [bench] mean: 182.4 ms, p99: 189 ms
```

## 9. Export for Edge Impulse Studio

For the production firmware build, upload the trained Keras model to Edge Impulse Studio and use the "Arduino library (Quantized int8)" export. This produces the `hatch_acoustic_inferencing` library that drops into `/firmware/lib/`. The exported library provides the `run_classifier()` symbol that `acoustic.cpp` calls.

Alternatively, link directly against TFLite-Micro and substitute `run_classifier_from_features()` with a direct interpreter invocation — this is what the open-source release does (no proprietary dependency).

## 10. Next steps — fine-tuning on Singapore-specific data

During Phase 3 of the grant project, each deployed node uploads short event-triggered clips. These are manually (and later semi-supervisedly) labelled and added to a Singapore-specific dataset. The fine-tuning loop:

1. Pre-train on HumBug (this notebook).
2. Periodically (monthly during Phase 3) fine-tune the last two conv blocks + dense head on `HumBug + SG-recorded` mixed dataset, with the SG portion oversampled to ~30% of each batch.
3. Re-quantize, re-deploy via OTA.

The fine-tuning protocol is in Notebook 04 (to be created when SG data is collected).

**Open-source release.** On project conclusion, the trained models, the full notebook chain, and the Singapore-specific dataset will be released under MIT (code) and CC-BY 4.0 (dataset).